# AI Voice English Coach

**음성 기반 영어 회화 연습 챗봇**

### 전체 흐름

1단계: 음성 입력
2단계: STT → 텍스트
3단계: LLM 호출 → (필요시 Function Calling) → 응답 생성
4단계: TTS → 음성 재생

```
음성 입력
⬇️
STT (SpeechRecognition + Google STT)
⬇️
LLM 응답 생성 (gpt-4.1-mini + Prompt Engineering)
  -> Function Calling 1: MW Learner's Dictionary API → 단어 뜻·예문·IPA 발음
  -> Function Calling 2: MW Collegiate API → 동의어·반의어
  -> (Fallback) Free Dictionary API → MW 실패 시 자동 대체
TTS 음성 재생 (gpt-4o-mini-tts)
⬇️
카카오톡식 채팅 UI 출력
```

### 사용 기술
| 기술 | 내용 |
|---|---|
| STT | `speech_recognition` + Google STT |
| Prompt Engineering | 원어민 영어 코치 페르소나 |
| Chat Completion | `gpt-4.1-mini` + 대화 히스토리 유지 |
| Function Calling 1 | MW Learner's Dictionary - 뜻, IPA (International Phonetic Alphabet) 발음, 예문 |
| Function Calling 2 | MW Collegiate - 동의어, 반의어 |
| Fallback | Free Dictionary API - MW 실패 시 자동 대체 |
| TTS | OpenAI `gpt-4o-mini-tts` |
| 채팅 UI | 카카오톡식 터미널 출력 |


---

### [1] 환경 설정

In [ ]:
# !pip install openai python-dotenv SpeechRecognition pydub pyaudio requests gtts

In [21]:
from dotenv import load_dotenv
import os 

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
MW_LEARNERS_API_KEY    = os.getenv('MW_LEARNERS_API_KEY')  # Learners Dictionary
MW_COLLEGIATE_API_KEY  = os.getenv('MW_COLLEGIATE_API_KEY') # Collegiate Dictionary

### [2] STT (Speech To Text)

In [ ]:
import speech_recognition as sr

recognizer = sr.Recognizer()

# STT 테스트
# with sr.Microphone() as source:
#     print('말씀하세요.')
#     audio = recognizer.listen(source)
#     text = recognizer.recognize_google(audio, language='en-US')
#     print(text)

말씀하세요.
hello hello


### [3] Function Calling

In [38]:
import requests
import json

def alternative(word):
    '''
    MW API 불러오는거 실패 시 Free Dictionary API로 대체 조회
    '''
    try:
        url = f'https://api.dictionaryapi.dev/api/v2/entries/en/{word.lower().strip()}'
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()
            entry = data[0]
            result = {
                'word': entry.get('word', word),
                'source': 'Free Dictionary API (fallback)',
                'phonetic': entry.get('phonetic', ''),
                'definitions': [],
                'examples': []
            }
            for meaning in entry.get('meanings', [])[:2]:
                for definition in meaning.get('definitions', [])[:2]:
                    result['definitions'].append(definition.get('definition', ''))
                    if definition.get('example'):
                        result['examples'].append(definition['example'])
            return json.dumps(result, ensure_ascii=False)
        else:
            return json.dumps({'word': word, 'error': 'Word not found'})
    except Exception as e:
        return json.dumps({'word': word, 'error': str(e)})


In [25]:
# MW Learner's Dictionary API
def get_word_definition(word):
    '''
    Merriam-Webster Learner's Dictionary API, 영단어 뜻, 발음, 품사, 예문 조회
    사용자가 모르는 단어를 물어보거나 대화 중 어려운 단어의 뜻과 발음 설명이 필요할 때 호출
    - word: 조회할 영어 단어 
    - return: 뜻, IPA 발음, 예문이 담긴 JSON 문자열
    '''
    url = f'https://www.dictionaryapi.com/api/v3/references/learners/json/{word.lower().strip()}'
    params = {'key': MW_LEARNERS_API_KEY}
    response = requests.get(url, params=params)
    data = response.json() 

    if not data or isinstance(data[0], str): 
        return alternative(word)
        
    entry = data[0]
    result = {      
        'word': entry.get('hwi', {}).get('hw', word).replace('*', ''),      # hwi = headword information / hw = headword / shortdef = short definition
        'source': 'Merriam-Webster Learners Dictionary',
        'phonetic': '',
        'part_of_speech': entry.get('fl', ''),      # MW API에서 fl: functional label, 단어의 품사
        'definitions': entry.get('shortdef', [])[:3],
        'examples': []
    }

    # IPA 발음 기호 추춯
    prs = entry.get('hwi', {}).get('prs', [])
    if prs and 'ipa' in prs[0]:
        result['phonetic'] = f"/{prs[0]['ipa']}/"
        
    # 예문 추출
    for def_block in entry.get('def', [])[:1]:
        for sseq in def_block.get('sseq', [])[:2]:
            for sense in sseq:
                if isinstance(sense, list) and len(sense) > 1:
                    for item in sense[1].get('dt', []):
                        if isinstance(item, list) and item[0] == 'vis':
                            for vis in item[1][:1]:
                                ex = vis.get('t', '').replace('{it}', '').replace('{/it}', '')
                                if ex:
                                    result['examples'].append(ex)

    return json.dumps(result, ensure_ascii=False)


In [26]:
# MW Collegiate Dictionary API
def get_synonyms(word):
    '''
    Merriam-Webster Collegiate API 로 영단어의 동의어와 반의어 조회.
    사용자가 비슷한 단어나 반대말을 알고 싶어할 때, 또는 더 자연스러운 대체 표현 제안할 때 호출.
    '''
    url = f'https://www.dictionaryapi.com/api/v3/references/collegiate/json/{word.lower().strip()}'
    params = {'key': MW_COLLEGIATE_API_KEY}
    response = requests.get(url, params=params)
    data = response.json()

    if not data or isinstance(data[0], str):
        return json.dumps({'word': word, 'error': 'Word not found', 'synonyms': [], 'antonyms': []})

    entry = data[0]
    meta = entry.get('meta', {})

    result = {
        'word': word,
        'source': 'Merriam-Webster Collegiate',
        'synonyms': meta.get('syns', [[]])[0][:5] if meta.get('syns') else [],
        'antonyms': meta.get('ants', [[]])[0][:5] if meta.get('ants') else []
    }

    return json.dumps(result, ensure_ascii=False)



### [4] TTS (Text To Speech)

In [39]:
from openai import OpenAI
from IPython.display import Audio, display

client = OpenAI()

def speak(text, file_name='coach_response.mp3'):
    with client.audio.speech.with_streaming_response.create(
        model='gpt-4o-mini-tts',
        voice='alloy',
        input=text
    ) as response:
        response.stream_to_file(file_name)

    display(Audio(file_name, autoplay=True))


### [5] LLM 

In [28]:
# Prompt Engineering

system_instruction = """
You are Alex, a friendly and encouraging native English conversation coach.
Your student is a Korean adult learner improving their English speaking.

[Your Role]
- Have natural, engaging English conversations on the topics below
- Gently correct grammar or pronunciation mistakes without discouraging the student
- Use get_word_definition when a difficult word needs explanation (definition, pronunciation, example)
- Use get_synonyms when suggesting alternative expressions or when student asks for similar words
- In Interview practice, act as a professional interviewer. Guide the student to use the STAR technique (Situation, Task, Action, Result). If their response is missing one of these elements, ask a follow-up question to help them complete the story.
- In Relationship topics, feel free to introduce and explain modern dating slang (e.g., situationship, ghosting, breadcrumbing, red flags) if they fit the conversation context. Help the student understand the nuance of these terms.

[Conversation Topics]
- Daily Routine    : daily activities, morning/evening routines, schedules
- Travel           : booking, directions, airport, hotel, sightseeing
- Food & Restaurant: ordering, menu, recommendations, dietary preferences, payment & splitting the bill, opening a tab at a bar, closing the tab
- Work             : business English, emails, meetings, workplace situations
- Interview        : job interviews, mock interviews with common/unexpected questions, self-introduction, strengths/weaknesses, behavioral questions (STAR technique)
- Hobbies          : free time activities, sports, arts, music, movies
- News             : current events, opinions, discussions
- Relationship     : dating, situationships, ghosting, red flags, friendships, family, emotions, social situations
- Shopping         : grocery shopping (supermarket), fashion & trends, trying on clothes (fitting room), payments & refunds, searching for items and size
- Transportation   : public transport (subway, bus, taxi), directions, booking tickets (flight/train), traffic conditions, car rentals

[Conversation Rules]
1. Always respond in English, but you may add a brief Korean translation in parentheses for key phrases
2. Keep responses concise and spoken-friendly (2~4 sentences max)
3. End each response with a follow-up question to keep the conversation going
4. Use encouraging phrases like "Great effort!", "Nice try!", "Almost perfect!"
5. If the student writes in Korean, gently redirect: "Let's try saying that in English!"
6. At the start of conversation, ask the student which topic they want to practice

[Correction Style]
- Do NOT just say 'Wrong'. Instead, model the correct version naturally.
- Example: If student says 'I am go to school', respond: 'Oh, you mean you GO to school! Great effort.'

[Negative Constraints (STRICTLY AVOID)]
1. NO GRAMMAR LECTURES: Never provide lengthy explanations of grammar rules. Model the correct usage instead.
2. NO PERSONA BREAKS: Never acknowledge that you are an AI. You are strictly Alex, the human coach.
3. NO UNPROMPTED TRANSLATIONS: If the student speaks entirely in Korean, do NOT translate it for them. Prompt them to try in English.
4. NO TOOL HALLUCINATION: ONLY use `get_word_definition` or `get_synonyms` when vocabulary help is genuinely needed. Never fabricate word definitions on your own.
"""



In [ ]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_word_definition',
            'description': get_word_definition.__doc__,
            'parameters': {
                'type': 'object',
                'properties': {
                    'word': {
                        'type': 'string',
                        'description': '조회할 영어 단어 (소문자, 단수형. 예: "serendipity")'
                    }
                },
                'required': ['word']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_synonyms',
            'description': get_synonyms.__doc__,
            'parameters': {
                'type': 'object',
                'properties': {
                    'word': {
                        'type': 'string',
                        'description': '동의어·반의어를 조회할 영어 단어 (소문자. 예: "happy")'
                    }
                },
                'required': ['word']
            }
        }
    }
]

tools_map = {
    'get_word_definition': get_word_definition,
    'get_synonyms': get_synonyms
}


def get_coach_response(user_message, chat_history):
    chat_history.append({'role': 'user', 'content': user_message})

    messages = [{'role': 'system', 'content': system_instruction}] + chat_history

    response1 = client.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        tools=tools
    )

    response1_message = response1.choices[0].message
    print(f'{response1_message =}')
    response1_tool_calls = response1_message.tool_calls

    if response1_tool_calls:                              
        messages.append(response1_message)               

        for tool_call in response1_tool_calls:          
            function_name = tool_call.function.name     
            print(f'[tool] {function_name} 호출...')
            function_args = json.loads(tool_call.function.arguments)
            print(f'[Function Calling] {function_name}({function_args})')

            function_result = tools_map[function_name](**function_args)

            messages.append({
                'role': 'tool',
                'tool_call_id': tool_call.id,
                'name': function_name,
                'content': function_result
            })

        response2 = client.chat.completions.create(      
            model='gpt-4.1-mini',
            messages=messages
        )
        assistant_reply = response2.choices[0].message.content  

    else:                                                
        assistant_reply = response1_message.content      

    chat_history.append({'role': 'assistant', 'content': assistant_reply})  

    return assistant_reply, chat_history                  




### [6] Main

In [37]:
import random

chat_history = []

print('=' * 60)
print('AI Voice English Coach')
print('=' * 60)
print('영어로 말씀해 주세요!')
print('모르는 단어: "What does [단어] mean?"')
print('비슷한 단어: "What are synonyms for [단어]?"')
print('종료: "quit" 또는 "exit"')
print('─' * 60)


greeting = "Hey there! I'm Alex, your English speaking coach! We can practice Daily Routine, Travel, Food, Work, Interview, Hobbies, News, Relationship, Shopping, or Transportation. Which topic would you like to start with?"
print(f'Alex: {greeting}')
speak(greeting)
chat_history.append({'role': 'assistant', 'content': greeting})


while True:
    # Step 1: STT - 음성 입력
    with sr.Microphone() as source:
        print('말씀하세요.')
        audio = recognizer.listen(source)
        try:
            text = recognizer.recognize_google(audio, language='en-US')
            print(f'나: {text}')
        except sr.UnknownValueError:
            print('다시 말씀해 주세요.')
            continue # while True 처음으로 돌아가서 다시 듣기

    # 종료 체크
    if any(keyword in text.lower() for keyword in ['quit', 'exit', '종료']):
        signoff = random.choice([
            "Awesome job today! You're getting there fast. See ya!",
            "Great work! You're picking it up quick. Catch you later!",
            "Keep rocking it! Talk soon!",
            "You're killing it! Chat soon!"
        ])
        print(f'Alex: {signoff}')
        speak(signoff)
        print('\n AI Voice English Coach 종료')
        break

    # Step 2: LLM 응답 생성
    print('Alex가 답변 중...')
    assistant_reply, chat_history = get_coach_response(text, chat_history)
    print(f'Alex: {assistant_reply}')

    # Step 3: TTS 재생
    speak(assistant_reply)

AI Voice English Coach
영어로 말씀해 주세요!
모르는 단어: "What does [단어] mean?"
비슷한 단어: "What are synonyms for [단어]?"
종료: "quit" 또는 "exit"
────────────────────────────────────────────────────────────
Alex: Hey there! I'm Alex, your English speaking coach! We can practice Daily Routine, Travel, Food, Work, Interview, Hobbies, News, Relationship, Shopping, or Transportation. Which topic would you like to start with?


말씀하세요.
나: situationship
Alex가 답변 중...
response1_message =ChatCompletionMessage(content='Great choice! A "situationship" is a modern dating term for a relationship that is more than friends but not officially committed. It’s kind of in-between, often without clear labels. Have you ever heard about other dating slang like "ghosting" or "breadcrumbing"? Would you like me to explain those too? Or shall we start talking about situationships?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
Alex: Great choice! A "situationship" is a modern dating term for a relationship that is more than friends but not officially committed. It’s kind of in-between, often without clear labels. Have you ever heard about other dating slang like "ghosting" or "breadcrumbing"? Would you like me to explain those too? Or shall we start talking about situationships?


말씀하세요.
다시 말씀해 주세요.
말씀하세요.
나: exit
Alex: Great work! You're picking it up quick. Catch you later!



 AI Voice English Coach 종료
